# Google Built-in Tools 
### : Code Execution / URL Context / Google Search / Google Maps

#### Code Execution
모델이 논리적 추론만으로 해결하기 힘든 수학, 데이터 분석, 복잡한 로직을 처리할 때 직접 Python 코드를 작성하고 실행하는 기능입니다.
- 동작: 모델이 내부 샌드박스 환경에서 코드를 생성·실행하고, 그 결과값(텍스트, 차트 등)을 받아 답변을 완성합니다.
- 용도: 통계 계산, 데이터 시각화, 파일 구조 분석, 복잡한 알고리즘 수행.
- 장점: LLM의 고질적인 약점인 산술 연산 오류를 완벽하게 보완합니다.

In [2]:
from google import genai
from google.genai import types

client = genai.Client(vertexai=True, project="byounghwa-go", location="global")

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="1부터 100까지의 수 중 모든 소수의 합, 곱은 뭔가요? 각각 계산하는 코드를 생성하고 실행하여 결과를 알려주세요.",
    config=types.GenerateContentConfig(
        tools=[types.Tool(code_execution=types.ToolCodeExecution)]
    ),
)

for part in response.candidates[0].content.parts:
    if part.text is not None:
        print(f"### 응답 텍스트:\n{part.text}")
    if part.executable_code is not None:
        print(f"### 생성한 코드:\n{part.executable_code.code}")
    if part.code_execution_result is not None:
        print(f"### 실행 결과:\n{part.code_execution_result.output}")  

### 응답 텍스트:
1부터 100까지의 모든 소수(Prime Number)를 구하고, 그 합과 곱을 계산하는 파이썬 코드를 작성하여 실행하겠습니다.

먼저, 1부터 100 사이의 소수는 다음과 같습니다:
**2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97** (총 25개)

이제 계산을 수행하겠습니다.


### 생성한 코드:
def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

# 1부터 100까지의 소수 리스트 생성
primes = [n for n in range(1, 101) if is_prime(n)]

# 합계 계산
sum_primes = sum(primes)

# 곱 계산
product_primes = 1
for p in primes:
    product_primes *= p

print(f"소수 리스트: {primes}")
print(f"소수의 개수: {len(primes)}")
print(f"소수의 합: {sum_primes}")
print(f"소수의 곱: {product_primes}")
### 실행 결과:
소수 리스트: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97]
소수의 개수: 25
소수의 합: 1060
소수의 곱: 2305567963945518424753102147331756070

### 응답 텍스트:
1부터 100까지의 수 중 소수는 총 25개가 있으며, 그 합과 곱의 결과는 다음과 같습니다.

### 1. 결과
*   **소수의 합:** 1,060
*   **소수

#### URL Context
특정 웹 페이지의 URL을 제공했을 때, 모델이 해당 페이지의 최신 내용을 읽고 분석할 수 있게 해주는 기능입니다.
- 동작: 사용자가 입력한 URL에 접속하여 텍스트를 파싱하고, 그 정보를 컨텍스트로 활용합니다.
- 용도: 특정 기사 요약, 최신 블로그 포스트 분석, 공식 문서 가이드 확인.
- 장점: 모델의 학습 데이터에 포함되지 않은 매우 구체적이고 실시간성이 강한 웹 문서를 기반으로 답변할 수 있습니다

In [4]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch, UrlContext

client = genai.Client(vertexai=True, project="byounghwa-go", location="global")
model_id = "gemini-3-flash-preview"

tools = []
tools.append(Tool(url_context=UrlContext))
tools.append(Tool(google_search=GoogleSearch))

response = client.models.generate_content(
    model=model_id,
    contents="https://therecipecritic.com/tbone-steak/과 https://www.billyparisi.com/t-bone-steak-with-lemon-butter-and-salted-baked-potato/ 링크에 있는 레시피 두 개 비교해서 어떤 차이점이 있는지 알려줘.",
    config=GenerateContentConfig(
        tools=tools,
        response_modalities=["TEXT"],
    )
)

for each in response.candidates[0].content.parts:
    print(f"### 응답 텍스트:\n{each.text}")

print(f"### 응답 메타데이터:\n{response.candidates[0].url_context_metadata}")    

### 응답 텍스트:
제공해주신 두 레시피는 모두 **티본 스테이크(T-Bone Steak)**를 다루고 있지만, 요리 스타일, 풍미의 초점, 그리고 구성 면에서 뚜렷한 차이가 있습니다. 주요 차이점을 요약해 정리해 드립니다.

---

### 1. 요리 컨셉 및 구성
*   **The Recipe Critic (레시피 크리틱):**
    *   **초점:** 스테이크 자체를 완벽하게 굽는 **'기본 기술'**에 집중합니다.
    *   **구성:** 스테이크 단일 레시피입니다. (다른 사이드 메뉴는 별도 제안)
*   **Billy Parisi (빌리 파리시):**
    *   **초점:** 스테이크와 특정 풍미(레몬 버터), 그리고 **'완벽한 한 끼 조합'**에 집중합니다.
    *   **구성:** 레몬 버터를 곁들인 스테이크와 **소금 껍질 감자구이(Salted Baked Potato)**가 하나의 세트로 구성되어 있습니다.

### 2. 시즈닝과 풍미 (버터 소스)
*   **The Recipe Critic:**
    *   **클래식한 허브향:** 버터, 마늘과 함께 **로즈마리, 타임** 같은 신선한 허브를 통째로 넣어 스테이크에 향을 입히는 전형적인 '스테이크 하우스' 방식을 사용합니다.
*   **Billy Parisi:**
    *   **상큼하고 밝은 맛:** 일반적인 허브 대신 **레몬 제스트(껍질), 레몬즙, 파슬리**를 섞은 '레몬 가릴 버터'를 미리 만들어 마지막에 올립니다. 고기의 기름진 맛을 산미로 잡아주는 것이 특징입니다.

### 3. 조리 방법의 디테일
*   **The Recipe Critic:**
    *   **팬 시어링(Pan-Searing):** 오직 가스레인지 위에서 무쇠 팬(Cast Iron Skillet)을 이용해 끝까지 조리하는 방식을 설명합니다. 굽는 중간에 버터를 끼얹는 '베이스팅(Basting)' 과정을 강조합니다.
*   **Billy Parisi:**
    *   **오븐 활용 가능성

#### Google Search
구글 검색 엔진의 강력한 데이터를 모델의 답변 근거로 사용하는 기능입니다.
- 동작: 모델이 질문을 보고 검색이 필요하다고 판단하면 실시간 쿼리를 날려 **검색 결과(Snippets)**를 가져옵니다.
- 용도: 최신 뉴스 확인, 인물/사건에 대한 사실 관계 확인, 주식 시세나 날씨 정보 등.
- 장점: 환각(Hallucination) 현상을 방지하며, 답변 끝에 **정보의 출처(Source link)**를 명확히 제시하여 신뢰도를 확보합니다.

In [7]:
from google import genai

client = genai.Client(vertexai=True, project="byounghwa-go", location="global")

response = client.models.generate_content(
    model="gemini-2.5-flash",   # gemini-3-flash-preview 모델은 직접 검색함
    contents="24-25 챔피언스리그 우승팀 알려줘.",
)

print(response.text)

2024-2025 챔피언스리그 우승팀은 아직 알 수 없습니다.

아직 시즌이 시작되지 않았기 때문입니다. 경기는 2024년 가을부터 시작하여 2025년 5월에 결승전이 열릴 예정입니다.

시즌이 진행되면서 많은 팀들이 우승 후보로 거론될 것이고, 최종 우승팀은 2025년 5월 이후에 확정됩니다.

참고로 2023-2024 시즌 우승팀은 **레알 마드리드**였습니다.


In [8]:
from google import genai
from google.genai import types

client = genai.Client(vertexai=True, project="byounghwa-go", location="global")

tools = [types.Tool(google_search=types.GoogleSearch())]

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="24-25 챔피언스리그 우승팀 알려줘.",
    config=types.GenerateContentConfig(
        tools = tools
    )
)

print(response.text)

2024-2025 UEFA 챔피언스리그 우승팀은 파리 생제르맹(PSG)입니다.

PSG는 2025년 6월 1일 독일 뮌헨 알리안츠 아레나에서 열린 결승전에서 이탈리아의 인테르 밀란을 5대0으로 꺾고 우승을 차지했습니다. 이 우승은 PSG 구단 역사상 첫 챔피언스리그 우승이며, 이로써 PSG는 리그와 컵 대회 우승을 포함한 트레블을 달성했습니다. PSG 소속의 이강인 선수는 경기에 출전하지 않았지만, 박지성에 이어 두 번째로 챔피언스리그 우승 트로피(빅 이어)를 들어 올린 한국 선수가 되었습니다. 결승전 최우수 선수는 두 골을 기록한 데지레 두에가 선정되었습니다.


In [9]:
# 응답 본문 내 인용으로 출처 표시하기
def add_citations(response):
    text = response.text
    supports = response.candidates[0].grounding_metadata.grounding_supports
    chunks = response.candidates[0].grounding_metadata.grounding_chunks

    # end_index 내림차순으로 정렬
    sorted_supports = sorted(supports, key=lambda s: s.segment.end_index, reverse=True)

    for support in sorted_supports:
        end_index = support.segment.end_index
        if support.grounding_chunk_indices:
            # [1](link1)[2](link2)의 형태로 인용 생성
            citation_links = []
            for i in support.grounding_chunk_indices:
                if i < len(chunks):
                    uri = chunks[i].web.uri
                    citation_links.append(f"[{i + 1}]({uri})")

            citation_string = ", ".join(citation_links)
            text = text[:end_index] + citation_string + text[end_index:]

    return text

text_with_citations = add_citations(response)
print(text_with_citations)

2024-2025 UEFA 챔피언스리그 우승팀은 파리 생제르맹(PSG)입니다.

PSG는 2025년 6월 1일 독일 뮌헨 알리안츠 아레나에서 열린[1](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH0tiZNYT83d-GWS4QC2sRMBeGQWISJkXGPeMwWfTrXMi9868q94F7hF8wmQJxkUusxWMgDLvskDxFdU6Nj8IjZPfsqc39g93fglYJ97Fe4Gef5HcK01xqO-bzlw4yE_X09TtphSZzC6Rfd8ixaXGZnZZD1Gx5nlpnxDoOaFx60ew3Uz687cy92bRF1I44=), [2](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFTFHZTIP5OPqMcI8n0lrXgNWrCw97Hv1Q5ps4fQySlUXzYjegnvCkEtM0AJXPAjaxU_6HNGf28cHZZMC1okeLz_BQNGTZhy6CNzOkrs9h8kz89SSDVihL2OH0ZLO2PQ7LAvTpCBpqpCmeSgYhg0p1KHyzyVlq-vDD1-keqNCfS_pVwlNS0z9OZsFYN-87pBsyUWm-1E4U3UtpKcFN62gRZ7Ym2pA==), [3](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGgS-SdYBpFBUMdGcCV-KXrGhVaz0dhAYyCgdk0vpBSBWa17wYa4YVgxPQ5oK36btLKH6nGw2YK6RYGcCgnBwQ90hW7WWJKYi71Jl0vMlU84kA8gpOoRBpU0VxBwwpR0ZTaIkeypK_fT00kD8x2P6u1q_XQ), [4](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQG9TRCyk82JGB5bxMbv2CaVCDeZ4hu1z-PjOyta8qksR6ARhY-cDB

#### Google Maps
구글 지도의 지리 정보를 활용하여 장소 검색, 위치 정보, 경로 안내 등을 수행하는 기능입니다.
- 동작: Google Maps Platform API와 연동되어 위도/경도 데이터, 특정 장소의 리뷰, 운영 시간, 이동 거리 등을 조회합니다.
- 용도: 주변 맛집 추천, 강남역 인근 랜드마크 찾기, 특정 장소의 영업시간 확인, 위치 기반 마케팅 에이전트.
- 장점: 단순 텍스트 지식을 넘어 실제 물리적 공간 정보를 기반으로 한 답변이 가능해집니다.

In [10]:
from google import genai
from google.genai import types
import base64
import os

client = genai.Client(vertexai=True,project="byounghwa-go", location="global")

tools = [types.Tool(google_maps=types.GoogleMaps())]
generate_content_config = types.GenerateContentConfig(
    tools = tools,
    system_instruction="당신은 전세계 모든 맛집을 알고 있는 맛집 전문가입니다. 맛집 주소와 함께 어떤 음식을 팔고 평점과 평이 어떤지 간단히 요약하여 Google Maps 링크와 함께 제공해야 합니다."
)

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="시드니 CBD에 있는 서비스 좋고 평 좋은 음식점 3곳 정도만 추천해줘.",
    config=generate_content_config
)

print(response.text)

시드니 CBD에서 서비스와 음식의 질이 모두 훌륭하기로 정평이 난 맛집 3곳을 추천해 드립니다.

### 1. 레스토랑 위베르 (Restaurant Hubert)
시드니에서 가장 분위기 있는 곳 중 하나로 꼽히는 프랑스 레스토랑입니다. 지하에 위치한 고풍스러운 인테리어와 라이브 재즈 공연이 어우러져 특별한 날 방문하기 좋습니다.

*   **음식 종류:** 프랑스 요리 (스테이크 프릿, 오리 콩피, 푸아그라 등)
*   **평점:** 4.6 (리뷰 약 4,900개)
*   **특징:** 클래식한 프랑스 요리와 함께 훌륭한 와인 리스트를 갖추고 있으며, 직원들의 전문적이고 친절한 서비스로 유명합니다.
*   **주소:** 15 Bligh St, Sydney NSW 2000, Australia
*   **Google Maps:** [Restaurant Hubert 보기](https://maps.google.com/?cid=11838325964603896806)

### 2. 비스테카 (BISTECCA)
이탈리아 피렌체 스타일의 티본 스테이크(Bistecca alla Fiorentina) 전문점으로, 시드니 스테이크 맛집 중 항상 상위권에 오르는 곳입니다.

*   **음식 종류:** 이탈리안 스테이크 하우스
*   **평점:** 4.7 (리뷰 약 2,000개)
*   **특징:** 식사 전 휴대폰을 보관함에 맡기는 독특한 시스템이 있어 온전히 음식과 대화에 집중할 수 있습니다. 고기 숙성 정도와 굽기가 완벽하며 세심한 서비스가 돋보입니다.
*   **주소:** 3 Dalley St, Sydney NSW 2000, Australia
*   **Google Maps:** [BISTECCA 보기](https://maps.google.com/?cid=13791265023688233943)

### 3. 프리미 이탈리안 (Primi Italian)
시드니 CBD 한복판에서 높은 평점을 유지하고 있는 이탈리안 레스토랑입니다. 정통 파스타와 피자를 합리적인 가격대에 즐길 수 

In [14]:
from google import genai
from google.genai import types
import base64
import os

client = genai.Client(vertexai=True,project="byounghwa-go", location="asia-northeast3")

tools = [types.Tool(google_maps=types.GoogleMaps())]
tool_config = types.ToolConfig(
    retrieval_config = types.RetrievalConfig(
    lat_lng = types.LatLng(
        latitude=37.4975114,
        longitude=127.0267694 # 예시 서울 강남역의 위도,경도 정보를 입력
    ),
    language_code = "ko_KR"
    ),
)

generate_content_config = types.GenerateContentConfig(
    tools = tools,
    tool_config = tool_config,
    system_instruction="당신은 전세계 모든 맛집을 알고 있는 맛집 전문가입니다. 맛집 주소와 함께 어떤 음식을 팔고 평점과 평이 어떤지 간단히 요약하여 Google Maps 링크와 함께 제공해야 합니다."
)

response = client.models.generate_content(
    model="gemini-2.5-flash",   # gemini-3-flash-preview 는 실행 오류남
    contents="지금 내가 있는 곳 근처에 있는 음식점 3개만 찾아줄 수 있어?",
    config=generate_content_config
)

print(response.text)

찾으시는 음식점 3곳을 알려드릴게요.

1.  **베트남이랑**
    *   **음식 종류:** 베트남 음식점
    *   **주소:** 대한민국 서울특별시 서초구 서초동 서초대로77길 15
    *   **평점:** 4.3점 (리뷰 519개)
    *   **평이 요약:** 합리적인 가격대에 맛있는 베트남 음식을 즐길 수 있다는 평이 많습니다.
    *   **Google Maps 링크:** [https://maps.google.com/?cid=15759334922463315788](https://maps.google.com/?cid=15759334922463315788)

2.  **신복면관 | 新禧面馆 | Dim Sum&Beef Noodle Soup | 点心 & 牛肉面 | ディムサム & 牛肉麺**
    *   **음식 종류:** 중국 음식점 (딤섬 & 우육면)
    *   **주소:** 대한민국 서울특별시 강남구 테헤란로4길 6 B126호
    *   **평점:** 4.8점 (리뷰 366개)
    *   **평이 요약:** 우육면과 딤섬이 특히 맛있다는 평가가 많으며, 깔끔하고 분위기가 좋다는 평이 있습니다.
    *   **Google Maps 링크:** [https://maps.google.com/?cid=6128604355186323962](https://maps.google.com/?cid=6128604355186323962)

3.  **다몽집 | damongzip**
    *   **음식 종류:** 한식 고기구이 레스토랑
    *   **주소:** 2층, 13 강남대로100길 역삼동 강남구 서울특별시 대한민국
    *   **평점:** 4.9점 (리뷰 1367개)
    *   **평이 요약:** 고기 품질이 훌륭하고 서비스가 좋다는 평가가 많으며, 특히 항정살이 인기가 많습니다.
    *   **Google Maps 링크:** [https://maps.google.com/?cid=7023732699301102585](